In [93]:
print ("ok")
%pwd

ok


'/Users/himanshuagarwal/Desktop/MedicalAPP'

In [94]:
import os 
os.chdir("/Users/himanshuagarwal/Desktop/MedicalAPP")
%pwd

'/Users/himanshuagarwal/Desktop/MedicalAPP'

In [95]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import DirectoryLoader

In [96]:
def load_pdf_files(data):
    loader = DirectoryLoader(data, glob="*.pdf",loader_cls=PyPDFLoader)

    documents = loader.load()
    return documents

In [97]:
extracted_data = load_pdf_files("data")

In [98]:
from typing import List
from langchain_core.documents import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of Document objects, return a new list of Document objects
    containing only 'source' in metadata and the original page_content.
    """
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={
                    "source": doc.metadata.get("source"),
                    "page": doc.metadata.get("page")
                }
            )
        )
    return minimal_docs

In [100]:
minimal_docs = filter_to_minimal_docs(extracted_data)
print(minimal_docs[1])

page_content='U.S. DEPARTMENT OF HEALTH AND HUMAN SERVICES
Substance Abuse and Mental Health Services Administration
Center for Substance Abuse Treatment
1 Choke Cherry Road
Rockville, MD 20857
Steven L. Batki, M.D.
Consensus Panel Chair
Janice F. Kauffman, R.N., M.P.H., LADC, CAS
Consensus Panel Co-Chair
Ira Marion, M.A.
Consensus Panel Co-Chair
Mark W. Parrino, M.P.A. 
Consensus Panel Co-Chair
George E. Woody, M.D.
Consensus Panel Co-Chair
A Treatment 
Improvement 
Protocol
TIP
43
Medication-Assisted Treatment
For Opioid Addiction in 
Opioid Treatment Programs' metadata={'source': 'data/Medical_book.pdf', 'page': 1}


In [101]:
def text_splitter(documents, chunk_size=500, chunk_overlap=20):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    text_chunk = text_splitter.split_documents(documents)
    return text_chunk

In [ ]:
text_chunks = text_splitter(minimal_docs)
for i, text_chunks in enumerate(text_chunks):
    text_chunks.metadata["chunk_id"] = i

Number of text chunks: 2358


In [67]:
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
def download_embeddings():
    model_name = "BAAI/bge-small-en-v1.5"
    embeddings = HuggingFaceBgeEmbeddings(model_name=model_name)
    return embeddings

embedding = download_embeddings()

In [68]:
vector = embedding.embed_query("hello")


In [69]:
print (len(vector))

384


In [70]:
from dotenv import load_dotenv
import os 
load_dotenv()

True

In [71]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY

In [72]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY
pc = Pinecone(api_key=pinecone_api_key)
pc

In [73]:
from pinecone import ServerlessSpec

index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,  # Dimension of the embeddings
        metric="cosine",  # Similarity metric
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)




In [74]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(documents=text_chunks, embedding=embedding, index_name=index_name)

In [75]:
mydoc = Document (
    page_content="This is my new document and i will be having 50 crore in next 3 years",
    metadata={"source": "my_source"}
)

mydoc2 = Document (
    page_content="I have 2 kids 1 daughter and 1 son",
    metadata={"source": "my_source"}
)

In [76]:
retriever = docsearch.as_retriever(search_type = "similarity", search_kwargs={"k": 2})

In [77]:
retriever_doc = retriever.invoke("what is acne")
retriever_doc



[Document(id='39740497-3e57-4cf8-8aa2-b1f6aa653e92', metadata={'source': 'data/Medical_book.pdf'}, page_content='Hepatitis\nHepatitis A\nHepatitis A is an important viral liver infection\nthat affects people who abuse drugs at higher\nrates than rates found in the general popula-\ntion. Hepatitis A can cause serious morbidity\nand mortality in patients already infected with\nhepatitis B virus (HBV) or hepatitis C virus\n(HCV). OTPs should screen for hepatitis A\nvirus (HAV) and provide vaccination services\nor referral to such services for individuals who\nare unexposed.\nHepatitis B'),
 Document(id='2bd3d95b-7e2d-462d-8111-6895322821a6', metadata={'source': 'data/Medical_book.pdf'}, page_content='Hepatitis\nHepatitis A\nHepatitis A is an important viral liver infection\nthat affects people who abuse drugs at higher\nrates than rates found in the general popula-\ntion. Hepatitis A can cause serious morbidity\nand mortality in patients already infected with\nhepatitis B virus (HBV) or h

In [84]:
# from langchain_openai import ChatOpenAI
# chatModel = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0.7, max_tokens=1000)
from langchain_ollama import ChatOllama

chatModel = ChatOllama(
    model="llama3.2",
    temperature=0
)



In [85]:
from langchain_core.prompts import ChatPromptTemplate

system_prompt = """
You are an AI assistant for answering medical questions.

Use only the provided context to answer the user's question.

If the answer is not present in the context, say:
"I don't know based on the provided medical documents."

Do not make up or assume any information.

Keep your answers:
- Accurate
- Concise
- Easy to understand

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}")
    ]
)

In [86]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [87]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "input": RunnablePassthrough(),
    }
    | prompt
    | chatModel
    | StrOutputParser()
)


In [91]:
response = rag_chain.invoke(
    "What is medication-assisted treatment?"
)

print(response)

Medication-Assisted Treatment (MAT) is a type of treatment that combines medications with counseling and behavioral therapies to treat opioid addiction. The goal of MAT is to help individuals manage withdrawal symptoms, reduce cravings, and support long-term recovery.


In [92]:
response = rag_chain.invoke(
    "how many children do I have?"
)

print(response)

You have 2 children, a daughter and a son.
